In [1]:
# [Cell 1] - Imports and Load GloVe
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense, Dropout, Concatenate
import ast

In [2]:
# [Cell 2] - Load Data
data_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\data_light_preprocessed.csv"
df = pd.read_csv(data_path)

In [7]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171820 entries, 0 to 171819
Data columns (total 22 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             171820 non-null  int64  
 1   text                   171820 non-null  object 
 2   happy                  171820 non-null  int64  
 3   sad                    171820 non-null  int64  
 4   disgusted              171820 non-null  int64  
 5   mad                    171820 non-null  int64  
 6   scared                 171820 non-null  int64  
 7   surprised              171820 non-null  int64  
 8   neutral                171820 non-null  int64  
 9   POS_Tags               171820 non-null  object 
 10  TF_IDF                 171820 non-null  object 
 11  Sentiment_Score        171820 non-null  float64
 12  Sentiment              171820 non-null  object 
 13  Sentiment_Summary      171820 non-null  object 
 14  Pretrained_Embeddings  171820 non-nu

In [3]:
df.head()

,Unnamed: 0,text,happy,sad,disgusted,mad,scared,surprised,neutral,POS_Tags,...,Sentiment,Sentiment_Summary,Pretrained_Embeddings,Custom_Embeddings,Avg_Word_Length,Lexical_Density,Flesch_Score,Exclamation_Marks,Capitalized_Words,Numerics
0,0,that game hurt.,0,1,0,0,0,0,0,"[('that', 'DT'), ('game', 'NN'), ('hurt', 'VBD...",...,Negative,TextBlob: Negative (-0.40) | VADER: Negative (...,"[0.042507488280534744, 0.03550424054265022, 0....",[-7.41447270e-01 -7.19539300e-02 7.51998365e-...,4.000000,0.666667,119.19,0,0,0
1,2,"you do right, if you do not care then fuck them!",0,0,0,0,0,0,1,"[('you', 'PRP'), ('do', 'VBP'), ('right', 'RB'...",...,Neutral,TextBlob: Negative (-0.11) | VADER: Positive (...,"[0.0651797354221344, 0.028579892590641975, -0....",[ 0.44395053 0.49131754 0.5076775 -0.820524...,3.272727,0.272727,111.07,1,0,0
2,3,man i love reddit.,1,0,0,0,0,0,0,"[('man', 'NN'), ('i', 'VBZ'), ('love', 'VBP'),...",...,Positive,TextBlob: Positive (0.50) | VADER: Positive (0...,"[-0.06888318061828613, -0.0226387158036232, 0....",[ 0.01417221 0.06746969 0.97807264 -0.812218...,3.500000,0.750000,92.80,0,0,0
3,4,"[name] was nowhere near them, he was by the fa...",0,0,0,0,0,0,1,"[('[', 'NN'), ('name', 'NN'), (']', 'NNP'), ('...",...,Neutral,TextBlob: Positive (0.10) | VADER: Neutral (0....,"[0.03852083906531334, 0.10962970554828644, -0....",[-0.08791067 -0.4164605 0.30734888 -0.560325...,3.800000,0.200000,95.17,0,0,0
4,5,right? considering it is such an important doc...,1,0,0,0,0,0,0,"[('right', 'RB'), ('?', '.'), ('considering', ...",...,Positive,TextBlob: Positive (0.23) | VADER: Positive (0...,"[-0.03342488035559654, 0.11335867643356323, -0...",[-0.30626807 0.10606792 0.54559064 0.094694...,4.954545,0.500000,68.77,1,0,0


In [10]:
# Load GloVe embeddings
def load_glove_embeddings(path):
    embeddings_index = {}
    with open(path, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    return embeddings_index

# Load GloVe
glove_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\glove.6B.300d.txt"
embedding_dim = 300  # Dimension of GloVe embeddings
glove_embeddings = load_glove_embeddings(glove_path)
print(f"Loaded {len(glove_embeddings)} word vectors.")

Loaded 400000 word vectors.


In [17]:
# [Cell 2] - Load and Preprocess Data
# Load data
data_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\data_light_preprocessed.csv"
df = pd.read_csv(data_path)

# Process text
max_sequence_length = 100
texts = df['text'].astype(str).tolist()

# Create tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
vocab_size = len(tokenizer.word_index) + 1

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(texts)
X_text = pad_sequences(sequences, maxlen=max_sequence_length, padding='post')

# Create embedding matrix
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    embedding_vector = glove_embeddings.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

# Process Sentiment Score
X_sentiment = df['Sentiment_Score'].values.reshape(-1, 1)

# Process POS Tags
def process_pos_tags(pos_tags_str):
    pos_tags = ast.literal_eval(pos_tags_str)
    return [tag for _, tag in pos_tags]

pos_tags_list = [process_pos_tags(tags) for tags in df['POS_Tags']]
pos_tokenizer = Tokenizer()
pos_tokenizer.fit_on_texts(pos_tags_list)
pos_sequences = pos_tokenizer.texts_to_sequences(pos_tags_list)
X_pos = pad_sequences(pos_sequences, maxlen=max_sequence_length, padding='post')

# Process TF-IDF
def process_tfidf(tfidf_str):
    try:
        # Convert string representation to dictionary
        tfidf_dict = ast.literal_eval(tfidf_str)
        # Convert dictionary values to array
        return list(tfidf_dict.values())
    except:
        return []

# Process TF-IDF and get the maximum length
tfidf_lists = [process_tfidf(tfidf) for tfidf in df['TF_IDF']]
max_tfidf_length = max(len(tfidf) for tfidf in tfidf_lists)

# Pad the TF-IDF vectors to the same length
X_tfidf = np.zeros((len(df), max_tfidf_length))
for i, tfidf in enumerate(tfidf_lists):
    X_tfidf[i, :len(tfidf)] = tfidf

print("TF-IDF shape:", X_tfidf.shape)

TF-IDF shape: (171820, 21)


In [19]:
# [Cell 3] - Train/Val/Test Split (70-20-10)
from sklearn.model_selection import train_test_split

# First split (70-30)
X_train_text, X_temp_text, \
X_train_sent, X_temp_sent, \
X_train_pos, X_temp_pos, \
X_train_tfidf, X_temp_tfidf, \
y_train, y_temp = train_test_split(
    X_text, X_sentiment, X_pos, X_tfidf, y,
    test_size=0.3,
    random_state=42
)

# Second split (20-10)
X_val_text, X_test_text, \
X_val_sent, X_test_sent, \
X_val_pos, X_test_pos, \
X_val_tfidf, X_test_tfidf, \
y_val, y_test = train_test_split(
    X_temp_text, X_temp_sent, X_temp_pos, X_temp_tfidf, y_temp,
    test_size=0.33,
    random_state=42
)

In [20]:
# [Cell 4] - Build Model
# Inputs
text_input = Input(shape=(max_sequence_length,), name='text_input')
sentiment_input = Input(shape=(1,), name='sentiment_input')
pos_input = Input(shape=(max_sequence_length,), name='pos_input')
tfidf_input = Input(shape=(X_tfidf.shape[1],), name='tfidf_input')

# Text branch with GloVe embeddings
embedding_layer = Embedding(
    vocab_size,
    embedding_dim,
    weights=[embedding_matrix],
    input_length=max_sequence_length,
    trainable=False
)(text_input)
text_rnn = SimpleRNN(64, return_sequences=True)(embedding_layer)
text_rnn = SimpleRNN(32)(text_rnn)

# POS tags branch
pos_embedding = Embedding(
    len(pos_tokenizer.word_index) + 1,
    32,
    input_length=max_sequence_length
)(pos_input)
pos_rnn = SimpleRNN(32, return_sequences=True)(pos_embedding)
pos_rnn = SimpleRNN(16)(pos_rnn)

# Process other features
sentiment_dense = Dense(8, activation='relu')(sentiment_input)
tfidf_dense = Dense(32, activation='relu')(tfidf_input)

# Combine all features
combined = Concatenate()([
    text_rnn,
    pos_rnn,
    sentiment_dense,
    tfidf_dense
])

# Dense layers
x = Dense(64, activation='relu')(combined)
x = Dropout(0.3)(x)
x = Dense(32, activation='relu')(x)
output = Dense(len(emotion_cols), activation='softmax')(x)

# Create and compile model
model = Model(
    inputs=[text_input, sentiment_input, pos_input, tfidf_input],
    outputs=output
)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model summary
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 text_input (InputLayer)        [(None, 100)]        0           []                               
                                                                                                  
 pos_input (InputLayer)         [(None, 100)]        0           []                               
                                                                                                  
 embedding_2 (Embedding)        (None, 100, 300)     9748500     ['text_input[0][0]']             
                                                                                                  
 embedding_3 (Embedding)        (None, 100, 32)      1472        ['pos_input[0][0]']              
                                                                                            

In [21]:
# [Cell 5] - Train Model
history = model.fit(
    [X_train_text, X_train_sent, X_train_pos, X_train_tfidf],
    y_train,
    validation_data=(
        [X_val_text, X_val_sent, X_val_pos, X_val_tfidf],
        y_val
    ),
    epochs=10,
    batch_size=32,
    verbose=1
)

Epoch 1/10
3759/3759 [==============================] - 119s 31ms/step - loss: 1.3792 - accuracy: 0.4671 - val_loss: 1.3622 - val_accuracy: 0.4742
Epoch 2/10
3759/3759 [==============================] - 128s 34ms/step - loss: 1.3596 - accuracy: 0.4776 - val_loss: 1.3629 - val_accuracy: 0.4720
Epoch 3/10
3759/3759 [==============================] - 122s 32ms/step - loss: 1.3559 - accuracy: 0.4790 - val_loss: 1.3566 - val_accuracy: 0.4732
Epoch 4/10
3759/3759 [==============================] - 121s 32ms/step - loss: 1.3533 - accuracy: 0.4794 - val_loss: 1.3619 - val_accuracy: 0.4743
Epoch 5/10
3759/3759 [==============================] - 126s 33ms/step - loss: 1.3505 - accuracy: 0.4810 - val_loss: 1.3565 - val_accuracy: 0.4717
Epoch 6/10
3759/3759 [==============================] - 126s 34ms/step - loss: 1.3485 - accuracy: 0.4821 - val_loss: 1.3561 - val_accuracy: 0.4753
Epoch 7/10
3759/3759 [==============================] - 123s 33ms/step - loss: 1.3507 - accuracy: 0.4808 - val_loss: 1

In [ ]:
# [Cell 6] - Evaluate Model
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Plot training history
plt.figure(figsize=(12, 4))

# Plot accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# Evaluate on test set
y_pred = model.predict([X_test_text, X_test_sent, X_test_pos, X_test_tfidf])
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test_classes, y_pred_classes, 
                          target_names=emotion_cols))

# Plot confusion matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test_classes, y_pred_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_cols,
            yticklabels=emotion_cols)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# [Cell 7] - Save Model
# Save the entire model (architecture, weights, and optimizer state)
model.save('rnn_v2.h5')



In [ ]:
# Optionally, save the tokenizers for future use
import pickle

# Save tokenizers
tokenizer_dict = {
    'text_tokenizer': tokenizer,
    'pos_tokenizer': pos_tokenizer
}

with open('tokenizers.pkl', 'wb') as f:
    pickle.dump(tokenizer_dict, f)

print("Model and tokenizers saved successfully!")